# 🔭 06 — Prévision 2023
## Holt-Winters + Prophet + Ensemble + Bootstrap IC

Ce notebook utilise les modèles entraînés sur 2018-2022 pour prévoir 2023 avec intervalles de confiance bootstrap.

## 1. 📦 Imports & Chargement

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet

print("✅ Imports OK")

In [ ]:
# Charger les données complètes
df = pd.read_csv("prepared_data.csv", index_col=0, parse_dates=True)
test = pd.read_csv("test_data.csv", index_col=0, parse_dates=True)

# Pour Prophet
df_prophet = df[["revenue"]].reset_index().rename(columns={"date": "ds", "revenue": "y"})

print(f"✅ Données chargées : {len(df)} mois (2018-2022)")
print(f"   Test 2022 : {len(test)} mois")

## 2. 📈 Holt-Winters 2023 + Bootstrap IC

In [ ]:
# Holt-Winters sur toutes les données
hw_full = ExponentialSmoothing(df["revenue"], trend="add",
                                seasonal="multiplicative", seasonal_periods=12,
                                damped_trend=True).fit(optimized=True, use_brute=True)

forecast_hw = hw_full.forecast(12)
forecast_hw.index = pd.date_range("2023-01-01", periods=12, freq="MS")

# Bootstrap des résidus pour IC
rng = np.random.default_rng(42)
residuals = hw_full.resid.dropna()
simulations = np.array([rng.choice(residuals, size=12, replace=True) for _ in range(1000)])
simulations += forecast_hw.values

ic_low = np.percentile(simulations, 2.5, axis=0)
ic_high = np.percentile(simulations, 97.5, axis=0)

print("✅ Holt-Winters 2023 prêt")
print(f"   Total 2023 estimé : {forecast_hw.sum()/1e6:.2f}M€")

## 3. 🔮 Prophet 2023 (corrigé)

In [ ]:
# Holidays (identique au notebook 04)
covid_events = []
for y, m in [(2020,3),(2020,4),(2020,5),(2020,6)]:
    covid_events.append({"holiday": "covid_severe", "ds": pd.Timestamp(f"{y}-{m:02d}-01"),
                         "lower_window": 0, "upper_window": 0})
for y, m in [(2020,7),(2020,8),(2020,9),(2020,10),(2020,11),(2020,12),
             (2021,1),(2021,2),(2021,3),(2021,4),(2021,5),(2021,6)]:
    covid_events.append({"holiday": "covid_moderate", "ds": pd.Timestamp(f"{y}-{m:02d}-01"),
                         "lower_window": 0, "upper_window": 0})
for y in range(2018, 2024):
    covid_events.append({"holiday": "christmas", "ds": pd.Timestamp(f"{y}-12-01"),
                         "lower_window": 0, "upper_window": 0})
holidays_df = pd.DataFrame(covid_events)

# Prophet sur toutes les données
prophet_full = Prophet(holidays=holidays_df, yearly_seasonality=True,
                       weekly_seasonality=False, daily_seasonality=False,
                       seasonality_mode="multiplicative",
                       changepoint_prior_scale=0.03,
                       seasonality_prior_scale=8.0,
                       holidays_prior_scale=5.0,
                       interval_width=0.95)
prophet_full.add_seasonality(name="monthly", period=30.5, fourier_order=3)
prophet_full.fit(df_prophet)

future = prophet_full.make_future_dataframe(periods=12, freq="MS", include_history=False)
forecast_prophet = prophet_full.predict(future)

print("✅ Prophet 2023 prêt")
print(f"   Total 2023 estimé : {forecast_prophet['yhat'].sum()/1e6:.2f}M€")

## 4. 🤝 Ensemble 2023 & Visualisation

In [ ]:
# Poids basés sur les MAPE 2022 (chargés depuis la comparaison)
try:
    comp = pd.read_csv("comparaison_finale.csv", index_col=0)
    hw_mape = comp.loc[comp.index.str.contains("Holt", case=False), "MAPE (%)"].values[0]
    prophet_mape = comp.loc[comp.index.str.contains("Prophet", case=False), "MAPE (%)"].values[0]
except:
    hw_mape, prophet_mape = 5.0, 7.0  # fallback

hw_w = 1.0 / hw_mape
prophet_w = 1.0 / prophet_mape
total_w = hw_w + prophet_w

ensemble_2023 = (hw_w / total_w) * forecast_hw.values + (prophet_w / total_w) * forecast_prophet["yhat"].values[:12]
ic_low_ens = (hw_w / total_w) * ic_low + (prophet_w / total_w) * forecast_prophet["yhat_lower"].values[:12]
ic_high_ens = (hw_w / total_w) * ic_high + (prophet_w / total_w) * forecast_prophet["yhat_upper"].values[:12]

# Graphique
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax = axes[0]
df["revenue"].plot(ax=ax, label="Historique 2018-2022", color="#1565C0", alpha=0.6, linewidth=1.5)
ax.plot(forecast_hw.index, forecast_hw.values, color="#F57C00", linewidth=2, linestyle="--", alpha=0.7, label="Holt-Winters")
ax.plot(forecast_prophet["ds"], forecast_prophet["yhat"], color="#7B1FA2", linewidth=2, linestyle=":", alpha=0.7, label="Prophet")
ax.plot(forecast_hw.index, ensemble_2023, color="#C62828", linewidth=3, label="✅ Ensemble 2023")
ax.fill_between(forecast_hw.index, ic_low_ens, ic_high_ens, alpha=0.2, color="#C62828", label="IC 95%")
ax.axvline(pd.Timestamp("2023-01-01"), color="gray", linestyle=":", linewidth=2)
ax.set_title("Prévision Revenue 2023 — Ensemble Pondéré", fontsize=14, fontweight="bold")
ax.set_ylabel("Revenue (€)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M€"))
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax2 = axes[1]
months = forecast_hw.index.strftime("%b")
x = range(12)
ax2.plot(x, ensemble_2023 / 1e6, "o-", color="#C62828", linewidth=2.5, markersize=8)
ax2.fill_between(x, ic_low_ens / 1e6, ic_high_ens / 1e6, alpha=0.2, color="#C62828", label="IC 95%")
for i, val in enumerate(ensemble_2023 / 1e6):
    ax2.text(i, val + 0.15, f"{val:.2f}", ha="center", fontsize=8, fontweight="bold")
ax2.set_xticks(x); ax2.set_xticklabels(months)
ax2.set_title("Détail Mensuel 2023", fontsize=13, fontweight="bold")
ax2.set_ylabel("Revenue (M€)"); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("🔭 Prévision 2023", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("forecast_2023.png", dpi=120, bbox_inches="tight")
plt.show()
print("💾 Graphique → forecast_2023.png")

# Tableau
print("\n📊 Prévisions mensuelles 2023 :")
print(f"  {'Mois':<15} {'Prévision':>10} {'IC Bas':>10} {'IC Haut':>10}")
print("  " + "-"*47)
total = 0
for i, date in enumerate(forecast_hw.index):
    total += ensemble_2023[i]
    print(f"  {date.strftime('%B %Y'):<15} {ensemble_2023[i]/1e6:>8.2f}M€ {ic_low_ens[i]/1e6:>8.2f}M€ {ic_high_ens[i]/1e6:>8.2f}M€")

print("  " + "-"*47)
print(f"  {'TOTAL 2023':<15} {total/1e6:>8.2f}M€")
print(f"  {'TOTAL 2022':<15} {test['revenue'].sum()/1e6:>8.2f}M€")
print(f"  {'Variation':<15} {(total/test['revenue'].sum()-1)*100:>+7.1f}%")

# Sauvegarder
forecast_df = pd.DataFrame({
    "date": forecast_hw.index, "revenue_pred": ensemble_2023,
    "ic_low": ic_low_ens, "ic_high": ic_high_ens,
    "holtwinters": forecast_hw.values, "prophet": forecast_prophet["yhat"].values[:12]
})
forecast_df.to_csv("forecast_2023.csv", index=False)
print("\n💾 Prévisions → forecast_2023.csv")